In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

# Infinite Sequences and the Sieve of Eratosthenes

TypeScript has *generators*, i.e. functions that produce a stream of values one at a time.  The <samp>OCaml</samp> counterpart of a generator is a *sequence*.  The module `Seq` of the standard library defines the type of sequences as follows:
```
type 'a t = unit -> 'a node
and  'a node = Nil | Cons of 'a * 'a t
```
Here, `'a` is a *type variable*: `int Seq.t` is the type of sequences of integers, `string Seq.t` is the type of sequences of strings, and so on.  The keyword `and` defines the two types `t` and `node` simultaneously, since each of them refers to the other.  A sequence is a function that takes the argument `()` and returns a *node*.  This node is either
- `Seq.Nil`, which indicates that the sequence is empty, or
- `Seq.Cons (x, rest)`, where `x` is the first element of the sequence and `rest` is the sequence of the remaining elements.

As `rest` is again a function, the remaining elements are only computed when `rest` is called: The elements of a sequence are computed *lazily*.  Therefore, a sequence can be infinite.

The function `count_from` takes one argument:
- `n` is an integer.

It returns the following *infinite stream* of natural numbers:
$$ [k \in \mathbb{N} \mid n \leq k] = [n,\; n+1,\; n+2,\; \cdots]$$

**Implementation:** When the sequence `count_from n` is called with the argument `()`, it returns the node `Seq.Cons (n, count_from (n + 1))`.  The recursive call `count_from (n + 1)` returns immediately, since its result is a function that has not been called yet.  Hence, the recursion does not run forever.  The standard library provides the same sequence as `Seq.ints n`.

In [ ]:
let rec count_from (n : int) : int Seq.t =
  fun () -> Seq.Cons (n, count_from (n + 1))

As a sequence is a function, the toplevel displays it as `<fun>`.

In [ ]:
let nat_nums = count_from 0

Unlike a TypeScript generator, an <samp>OCaml</samp> sequence has no state: Every time a sequence is traversed, its elements are computed anew, starting with the first one.  In order to fetch the elements of a sequence one at a time, we need a *dispenser*.  The function `Seq.to_dispenser` turns a sequence into a dispenser, i.e. into a function of type `unit -> int option`.  Every call of this function returns the next element `x` of the sequence as `Some x`, or `None` if there are no more elements.  This is the counterpart of the method `next` of a TypeScript generator.  The dispenser `next_nat` stores internally how many elements it has already returned.

In [ ]:
let next_nat = Seq.to_dispenser nat_nums

In [ ]:
next_nat ()

In [ ]:
next_nat ()

The function `print_first` takes two arguments:
- `s` is a sequence of integers,
- `n` is a natural number.

It prints the first `n` numbers of the sequence `s`, one number per line.  The function `Seq.take n s` returns the sequence of the first `n` elements of `s`, and the function call `Seq.iter f s` calls the function `f` for every element of the sequence `s`.

In [ ]:
let print_first (s : int Seq.t) (n : int) : unit =
  Seq.iter (fun x -> Printf.printf "%d\n" x) (Seq.take n s)

In [ ]:
print_first (count_from 7) 3

The function `filter_stream` takes two arguments:
  * `s` is a sequence of integers,
  * `p` is a predicate, i.e. a function that maps integers to truth values.

It returns a new sequence that contains only those values `x` from `s`
such that `p x` is `true`.
$$ \texttt{filter_stream}(S, p) = [ x \in S \mid p(x) ] $$
The standard library provides the same functionality as `Seq.filter p s`.  We define `filter_stream` nevertheless in order to show how a sequence is processed by pattern matching.

**Implementation:** The result is a function that expects the argument `()`.  When it is called, it calls `s ()` in order to get the first node of `s`.
- If `s` is empty, the result is empty, too.
- If the first element `x` of `s` satisfies `p`, then `x` is the first element of the result, and the remaining elements are computed by filtering `rest`.
- Otherwise, `x` is skipped and the first node of the filtered sequence `filter_stream rest p` is returned.  As this sequence is a function, it has to be called with the argument `()`.

If `s` is infinite and none of its elements satisfies `p`, the search for the first element of the result runs forever.  This does not happen in this notebook.

In [ ]:
let rec filter_stream (s : int Seq.t) (p : int -> bool) : int Seq.t =
  fun () ->
    match s () with
    | Seq.Nil            -> Seq.Nil
    | Seq.Cons (x, rest) ->
        if p x then Seq.Cons (x, filter_stream rest p)
        else filter_stream rest p ()

Let us test `filter_stream` by creating a stream of numbers divisible by $3$.

In [ ]:
let div_by_3 = filter_stream (count_from 1) (fun n -> n mod 3 = 0);;
print_first div_by_3 5

The function `filter_prime` takes one argument:
- `s` is a sequence of integers.

It returns the first element `p` of `s`, filters all subsequent multiples of `p`, and calls `filter_prime` recursively on the resulting sequence.
$$ \texttt{filter_prime}(p :: R) =
  p :: \texttt{filter_prime}\bigl([ x \in R \mid x \;\texttt{mod}\; p \not= 0]\bigr) $$
If the elements of `s` are the numbers $2, 3, 4, \cdots$, then the first element of every sequence that `filter_prime` is called with is a prime number, since all multiples of smaller prime numbers have been removed before.

**Implementation:** As in `filter_stream`, the result is a function that expects the argument `()` and calls `s ()`.  As the recursive call of `filter_prime` returns a function, it is only evaluated further when the second element of the result is needed.  Therefore, the recursion does not run forever.

In [ ]:
let rec filter_prime (s : int Seq.t) : int Seq.t =
  fun () ->
    match s () with
    | Seq.Nil            -> Seq.Nil
    | Seq.Cons (p, rest) ->   (* p is the first number, it is prime *)
        (* remove all multiples of the prime p *)
        Seq.Cons (p, filter_prime (filter_stream rest (fun x -> x mod p <> 0)))

The value `primes` is the infinite sequence of *prime numbers*, which is generated using
the [*Sieve of Eratosthenes*](https://en.wikipedia.org/wiki/Sieve_of_Eratosthenes).  As no element of a sequence is computed before it is needed, `primes` can be defined as a value: Unlike in TypeScript, we need no function to create the sequence.

In [ ]:
let primes : int Seq.t = filter_prime (count_from 2)

In [ ]:
print_first primes 100